In [2]:
from datasets import load_dataset
import os, re
import opencc
from tqdm import tqdm
from dotenv import load_dotenv

from typing import Optional

load_dotenv()
os.makedirs("corpus", exist_ok=True)
cc = opencc.OpenCC('t2s')

In [3]:
def remove_wiki_artifacts(text: str) -> str:
    """Remove Wikipedia-specific template markers, reference tags, and navigation boxes."""
    # Remove reference markers and section headers that appear as standalone content
    text = re.sub(r'\s*(参考文献|参考资料|延伸阅读|外部连结|参见|注释|来源)\s*$', '', text)
    
    # Remove trailing Latin species names or template codes (e.g., "panamensis", "D", "Lin")
    # These are typically 1-3 uppercase/lowercase letters at end of line, not part of normal prose
    text = re.sub(r'\s+[A-Za-z]{1,4}\s*$', '', text)
    
    # Remove coordinate stubs like "赤经，赤纬" or "北纬，东经" at end
    text = re.sub(r'\s*[赤南北纬东西经度分秒，,]+\s*$', '', text)
    
    # Remove disambiguation page markers: lines starting with "X可以指：" or ending with "消歧义"
    if re.match(r'^[^。！？；\n]+可以[：:]', text) or text.rstrip().endswith('消歧义'):
        return None  # Filter out entire disambiguation entries
    
    # Remove category/navigation tags at end (e.g., "中国武术", "小行星带天体", "利沃夫州村落")
    # These are typically 2-8 Chinese characters with no punctuation, often multiple space-separated
    trailing_tags = re.findall(r'[\u4e00-\u9fa5]{2,8}(?:\s+[\u4e00-\u9fa5]{2,8})*\s*$', text)
    if trailing_tags and not re.search(r'[。！？；]', text):
        # If line has no sentence-ending punctuation and ends with tag-like content, remove tags
        text = re.sub(r'\s+[\u4e00-\u9fa5]{2,8}(?:\s+[\u4e00-\u9fa5]{2,8})*\s*$', '', text)
    
    return text.strip()

def is_semantically_valid(text: str, min_content_ratio: float = 0.7) -> bool:
    """Check if text contains meaningful prose vs. template/category noise."""
    if not text or len(text) < 20:
        return False
    
    # Count Chinese characters vs. noise patterns
    zh_chars = len(re.findall(r'[\u4e00-\u9fa5]', text))
    total_chars = len(re.sub(r'\s+', '', text))
    
    if total_chars == 0:
        return False
    
    # Require minimum ratio of Chinese content
    if zh_chars / total_chars < min_content_ratio:
        return False
    
    # Filter lines that are mostly punctuation-separated short phrases (category lists)
    phrase_list = [p.strip() for p in re.split(r'[，,、;；]', text) if p.strip()]
    if len(phrase_list) >= 4 and all(len(p) <= 6 for p in phrase_list):
        # Likely a category/tag list, not prose
        return False
    
    # Require at least one sentence-ending punctuation for Wikipedia prose
    if not re.search(r'[。！？]', text):
        return False
    
    return True

def clean_wiki_line(text: str) -> Optional[str]:
    """Minimal cleaning: remove obvious artifacts, keep valid prose."""
    # 1. Basic whitespace normalization
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 2. Length filter (primary gate)
    if not (20 <= len(text) <= 200):
        return None
    
    # 3. Remove obvious noise patterns (non-destructive)
    # Remove trailing Latin codes like "panamensis", "D", "Lin" (1-4 letters at end)
    text = re.sub(r'\s+[A-Za-z]{1,4}\s*$', '', text)
    
    # Remove trailing coordinate stubs like "赤经，赤纬" or "北纬，东经"
    text = re.sub(r'\s*[赤南北纬东西经度分秒，,]+\s*$', '', text)
    
    # Remove standalone reference markers at end
    text = re.sub(r'\s*(参考文献|参考资料|延伸阅读|外部连结|参见|注释|来源)\s*$', '', text)
    
    # 4. Filter disambiguation pages (lines starting with "X可以指：")
    if re.match(r'^[^。！？；\n]{1,30}可以[：:]', text):
        return None
    
    # 5. Re-check length after artifact removal
    if len(text) < 20:
        return None
    
    # 6. Ensure sentence ending (append, don't reject)
    if not re.search(r'[。！？；\n]$', text):
        text += '。'
    
    return text

## Download & Clean Dataset

In [4]:

RAW_PATH = "corpus/raw_wiki.txt"
ds = load_dataset("wikimedia/wikipedia", "20231101.zh", split="train")

print("Extracting and cleaning Wikipedia text...")
valid_lines = []
for item in tqdm(ds, desc="Processing Wiki"):
    raw_text = item["text"]
    
    # Split into candidate sentences using Wikipedia's typical punctuation
    candidates = re.split(r'[。！？；\n]+', raw_text)
    
    for sent in candidates:
        cleaned = clean_wiki_line(sent)
        if cleaned:
            valid_lines.append(cleaned)

# Write pre-conversion text first
with open(RAW_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join(valid_lines) + '\n')
print(f"Raw Wikipedia text saved: {len(valid_lines)} lines -> {RAW_PATH}")

'[Errno 101] Network is unreachable' thrown while requesting HEAD https://huggingface.co/datasets/wikimedia/wikipedia/resolve/main/README.md
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.

In [ ]:
print("Executing batch traditional-to-simplified conversion (single OpenCC call)...")
with open(RAW_PATH, 'r', encoding='utf-8') as f:
    full_text = f.read()

simp_text = cc.convert(full_text)  # Single global call for performance

with open(RAW_PATH, 'w', encoding='utf-8') as f:
    f.write(simp_text)
print("Batch conversion complete. Ready for Stage One alignment.")

Executing batch traditional-to-simplified conversion (single OpenCC call)...
Batch conversion complete. Ready for Stage One alignment.
